# LeetCode 2644 — Find the Maximum Divisibility Score bằng Python

Link đề: https://leetcode.com/problems/find-the-maximum-divisibility-score/ · Độ khó: **Easy** · Chủ đề: Array

Skeleton LeetCode:

```python
class Solution:
    def maxDivScore(self, nums: List[int], divisors: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu khái niệm **"điểm chia hết" (divisibility score)** của một số `d`: đếm xem có bao nhiêu phần tử trong `nums` **chia hết cho `d`**.
2. Dùng toán tử `%` để kiểm tra chia hết (`x % d == 0`), biết rằng `True` / `False` trong Python cộng được như `1` / `0` nên `sum(x % d == 0 for x in nums)` chính là phép đếm.
3. Xử lý đúng **quy tắc hoà (tie-break)**: nhiều ước cùng điểm cao nhất thì trả về ước **nhỏ nhất** — bằng so sánh có điều kiện, bằng sắp xếp trước, hoặc bằng khóa `(điểm, -ước)`.
4. Viết 5 cách: hai vòng lặp đếm tay, `sum` + cập nhật kỷ lục có tie-break, sắp xếp `divisors` trước, khóa tuple với `max`, và cách tối ưu dùng `Counter` + bỏ qua các ước vô ích.
5. Tránh các lỗi: quên tie-break, chọn ước **lớn** nhất khi hoà, nhầm `x % d` với `d % x` (vai trò số bị chia và số chia), và khởi tạo kỷ lục `0` làm hỏng trường hợp mọi điểm đều `0`.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `nums` và `divisors` (chỉ số từ 0).

**Điểm chia hết** của `divisors[i]` là **số chỉ số `j`** sao cho `nums[j]` chia hết cho `divisors[i]`.

Hãy trả về phần tử `divisors[i]` có **điểm chia hết lớn nhất**. Nếu có nhiều phần tử cùng điểm lớn nhất, trả về **phần tử nhỏ nhất** trong số đó.

| Ví dụ | nums, divisors | Output | Giải thích |
|---|---|---|---|
| 1 | `[2,9,15,50]`, `[5,3,7,9]` | `3` | Điểm của `5` là `2` (`15`, `50`); của `3` là `2` (`9`, `15`); của `7` là `0`; của `9` là `1` (`9`). Max `2` ở `5` và `3` ⇒ chọn nhỏ hơn: `3`. |
| 2 | `[4,7,9,3,9]`, `[5,2,3]` | `3` | `5` chia hết 0 số; `2` chia hết `4` ⇒ 1; `3` chia hết `9, 3, 9` ⇒ 3. Lớn nhất là `3`. |
| 3 | `[12]`, `[10,16]` | `10` | Cả `10` và `16` đều không chia hết `12` ⇒ cùng điểm `0`; chọn nhỏ hơn: `10`. |

(Nếu bạn cần chắc chắn ví dụ 3, hãy đối chiếu lại với đề gốc trên LeetCode; ở đây ta đã kiểm tra bằng định nghĩa: `12 % 10 = 2`, `12 % 16 = 12`.)

Constraints:

- `1 <= nums.length, divisors.length <= 1000`
- `1 <= nums[i], divisors[i] <= 10^9`

In [1]:
nums, divisors = [2, 9, 15, 50], [5, 3, 7, 9]
for d in divisors:
    chia_het = [x for x in nums if x % d == 0]
    print(f"d={d}: các số chia hết cho d = {chia_het} -> điểm = {len(chia_het)}")

d=5: các số chia hết cho d = [15, 50] -> điểm = 2
d=3: các số chia hết cho d = [9, 15] -> điểm = 2
d=7: các số chia hết cho d = [] -> điểm = 0
d=9: các số chia hết cho d = [9] -> điểm = 1


## 3. Kiến thức nền

### 3.1. Chia hết và toán tử `%`

`a % b` là **số dư** khi chia `a` cho `b`. `a` **chia hết cho** `b` khi và chỉ khi `a % b == 0`. Ví dụ `15 % 5 = 0` (chia hết), `15 % 4 = 3`.

Cẩn thận **thứ tự**: bài hỏi "`nums[j]` chia hết cho `divisors[i]`" nên là `nums[j] % divisors[i] == 0`, tức **số bị chia ở bên trái**.

In [2]:
print(15 % 5, 15 % 4)          # 0 3
print(50 % 5 == 0)             # True: 50 chia hết cho 5
print(5 % 50 == 0, 5 % 50)     # False 5 : đảo vai trò là sai

0 3
True
False 5


### 3.2. `True` / `False` cộng được như `1` / `0`

Trong Python `bool` là kiểu con của `int`: `True == 1`, `False == 0`. Vì vậy `sum([True, False, True])` là `2`. Từ đó, số phần tử thoả điều kiện = `sum(điều kiện for x in ...)`.

In [3]:
print(True + True, True + False)            # 2 1
print(sum([True, False, True]))             # 2
nums = [2, 9, 15, 50]
print(sum(x % 5 == 0 for x in nums))        # 2: số phần tử chia hết cho 5

2 1
2
2


### 3.3. Biểu thức sinh (generator) trong `sum`

`sum(x % d == 0 for x in nums)` không có dấu `[]` nên **không** tạo danh sách trung gian, tiết kiệm bộ nhớ. Nó tương đương với vòng `for` đếm biến `count += 1` nhưng ngắn gọn.

In [4]:
nums = [4, 7, 9, 3, 9]
d = 3
# cách viết dài
count = 0
for x in nums:
    if x % d == 0:
        count += 1
print(count)                                  # 3
# cách viết ngắn
print(sum(x % d == 0 for x in nums))          # 3

3
3


### 3.4. Cập nhật "kỷ lục" với hai tiêu chí

Ta cần ước có **điểm cao nhất**, nếu hoà thì **ước nhỏ nhất**. Điều kiện cập nhật kỷ lục:

```python
if score > best_score or (score == best_score and d < best_d):
    best_score, best_d = score, d
```

Hai vế của `or`: hoặc điểm **hơn hẳn**, hoặc điểm **bằng** mà ước **nhỏ hơn**.

In [5]:
best_score, best_d = -1, None
for score, d in [(2, 5), (2, 3), (0, 7), (1, 9)]:
    if score > best_score or (score == best_score and d < best_d):
        best_score, best_d = score, d
    print(f"xét (score={score}, d={d}) -> kỷ lục hiện tại: score={best_score}, d={best_d}")

xét (score=2, d=5) -> kỷ lục hiện tại: score=2, d=5
xét (score=2, d=3) -> kỷ lục hiện tại: score=2, d=3
xét (score=0, d=7) -> kỷ lục hiện tại: score=2, d=3
xét (score=1, d=9) -> kỷ lục hiện tại: score=2, d=3


### 3.5. `sorted`, `min`, `max` với `key`; `Counter`

- `sorted(divisors)` trả bản sao **tăng dần**.
- `max(divisors, key=f)` trả phần tử có `f(d)` lớn nhất; hoà thì lấy phần tử **đứng trước**.
- `collections.Counter(nums)` đếm số lần xuất hiện của mỗi giá trị: `Counter([9, 3, 9])` là `{9: 2, 3: 1}`. Có ích khi `nums` có nhiều số trùng.

In [6]:
from collections import Counter
print(sorted([5, 3, 7, 9]))                         # [3, 5, 7, 9]
print(max([5, 3, 7, 9], key=lambda d: d % 3))       # 5 (5 % 3 = 2 lớn nhất)
print(Counter([4, 7, 9, 3, 9]))                     # Counter({9: 2, 4: 1, 7: 1, 3: 1})

[3, 5, 7, 9]
5
Counter({9: 2, 4: 1, 7: 1, 3: 1})


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tưởng tượng mỗi `d` trong `divisors` là một "thí sinh", còn mỗi số trong `nums` là một "giám khảo". Giám khảo `x` bỏ phiếu cho thí sinh `d` nếu `d` **chia hết** `x`. Điểm của thí sinh là số phiếu. Ta chọn thí sinh nhiều phiếu nhất; hoà thì chọn số nhỏ hơn.

Tính tay ví dụ 1 (`nums = [2, 9, 15, 50]`, `divisors = [5, 3, 7, 9]`):

- `d = 5`: `2 % 5 = 2`, `9 % 5 = 4`, `15 % 5 = 0` ✓, `50 % 5 = 0` ✓ ⇒ điểm `2`.
- `d = 3`: `2 % 3 = 2`, `9 % 3 = 0` ✓, `15 % 3 = 0` ✓, `50 % 3 = 2` ⇒ điểm `2`.
- `d = 7`: `2, 9, 15, 50` mod `7` là `2, 2, 1, 1` ⇒ điểm `0`.
- `d = 9`: chỉ `9 % 9 = 0` ✓ ⇒ điểm `1`.

Điểm lớn nhất là `2`, đạt bởi `5` và `3`; ước nhỏ hơn là `3`.

In [7]:
nums, divisors = [2, 9, 15, 50], [5, 3, 7, 9]
scores = {d: sum(x % d == 0 for x in nums) for d in divisors}
print("điểm từng ước:", scores)
top = max(scores.values())
print("điểm cao nhất:", top, "-> các ước đạt:", [d for d in divisors if scores[d] == top], "-> chọn nhỏ nhất:", min(d for d in divisors if scores[d] == top))

điểm từng ước: {5: 2, 3: 2, 7: 0, 9: 1}
điểm cao nhất: 2 -> các ước đạt: [5, 3] -> chọn nhỏ nhất: 3


## 5. Cách 1 — Brute Force / baseline: hai vòng lặp, đếm tay

**Ý tưởng:** với mỗi ước `d`, duyệt mọi `x` trong `nums` đếm số lần `x % d == 0`. So sánh với kỷ lục: điểm **cao hơn** thì cập nhật; điểm **bằng** nhưng `d` **nhỏ hơn** cũng cập nhật.

Khởi tạo `best_score = -1` (nhỏ hơn mọi điểm thật, kể cả `0`) để ước đầu tiên luôn thành kỷ lục.

In [8]:
from typing import List

class SolutionNested:
    def maxDivScore(self, nums: List[int], divisors: List[int]) -> int:
        best_score, best_d = -1, 0
        for d in divisors:
            score = 0
            for x in nums:
                if x % d == 0:
                    score += 1
            if score > best_score or (score == best_score and d < best_d):
                best_score, best_d = score, d
        return best_d

In [9]:
sol = SolutionNested()
print(sol.maxDivScore([2, 9, 15, 50], [5, 3, 7, 9]))      # 3
print(sol.maxDivScore([4, 7, 9, 3, 9], [5, 2, 3]))        # 3
print(sol.maxDivScore([12], [10, 16]))                    # 10

3
3
10


**Độ phức tạp:** `n = len(nums)`, `k = len(divisors)`. Hai vòng lồng nhau ⇒ Time `O(n · k)` (tối đa `10^6` phép `%`); Space `O(1)`.

**Nút thắt:** `10^6` phép chia vẫn rất nhanh; không cần cải tiến về bậc. Nhưng ta có thể **viết gọn** hơn, **đặt tie-break rõ ràng hơn** (sắp xếp trước, hoặc khóa tuple) và **bỏ việc thừa** (ước lặp, ước quá lớn).

## 6. Ý tưởng thuật toán / bước đột phá

1. **Đếm bằng `sum`.** `sum(x % d == 0 for x in nums)` thay vòng trong.
2. **Tie-break qua sắp xếp.** Nếu duyệt `divisors` theo thứ tự **tăng dần** và chỉ cập nhật khi `score > best_score` (chặt), thì khi hoà ước **nhỏ hơn đã nằm trước** và được giữ ⇒ không cần so sánh `d < best_d`.
3. **Tie-break qua khóa tuple.** Muốn "điểm lớn, ước nhỏ" ⇒ lấy `max` theo `(điểm, -ước)`: so sánh điểm trước, nếu bằng thì `-d` lớn hơn tức `d` nhỏ hơn.
4. **Bỏ việc thừa.** Một ước `d > max(nums)` không thể chia hết số dương nào của `nums` (vì số dương chia hết cho `d` phải `>= d`) ⇒ điểm `0`. Ước lặp lại chỉ cần tính một lần. `nums` có nhiều giá trị trùng ⇒ gom bằng `Counter`.

In [10]:
nums = [4, 7, 9, 3, 9]
print("max(nums) =", max(nums))
print("ước 100 có chia hết số nào không?", [x for x in nums if x % 100 == 0])   # [] vì 100 > mọi x

max(nums) = 9
ước 100 có chia hết số nào không? []


## 7. Các cách tiếp theo

### Cách 2 — `sum` + cập nhật kỷ lục có điều kiện hoà

Giống Cách 1 nhưng vòng trong thay bằng `sum`. Điều kiện cập nhật giữ nguyên: `score > best_score` hoặc (`score == best_score` và `d < best_d`).

In [11]:
class SolutionSum:
    def maxDivScore(self, nums: List[int], divisors: List[int]) -> int:
        best_score, best_d = -1, 0
        for d in divisors:
            score = sum(x % d == 0 for x in nums)
            if score > best_score or (score == best_score and d < best_d):
                best_score, best_d = score, d
        return best_d

print(SolutionSum().maxDivScore([2, 9, 15, 50], [5, 3, 7, 9]))     # 3
print(SolutionSum().maxDivScore([12], [10, 16]))                   # 10

3
10


**Độ phức tạp:** Time `O(n · k)`, Space `O(1)`.

### Cách 3 — Sắp xếp `divisors` rồi chỉ cập nhật khi `>` chặt

Sau `sorted(divisors)`, các ước tăng dần. Khi gặp điểm **bằng** kỷ lục, ước hiện tại **lớn hơn** ước đã lưu, nên ta **không** cập nhật — ước nhỏ hơn được giữ tự nhiên.

In [12]:
class SolutionSorted:
    def maxDivScore(self, nums: List[int], divisors: List[int]) -> int:
        best_score, best_d = -1, 0
        for d in sorted(divisors):
            score = sum(x % d == 0 for x in nums)
            if score > best_score:             # chặt: hoà thì giữ ước nhỏ hơn đã có
                best_score, best_d = score, d
        return best_d

print(SolutionSorted().maxDivScore([2, 9, 15, 50], [5, 3, 7, 9]))   # 3
print(SolutionSorted().maxDivScore([12], [10, 16]))                 # 10

3
10


**Độ phức tạp:** sắp xếp `O(k log k)` + `O(n · k)` ⇒ Time `O(n · k)` (vì `n` và `k` cùng cỡ `10^3`, `k log k` nhỏ hơn nhiều); Space `O(k)` cho bản sao đã sắp xếp.

### Cách 4 — `max` với khóa tuple `(điểm, -ước)`

Một dòng: `max(divisors, key=lambda d: (score(d), -d))`. Tuple lớn nhất là tuple có điểm lớn nhất và, khi hoà, `-d` lớn nhất ⇒ `d` nhỏ nhất. Tie-break nằm ngay trong khóa nên không phụ thuộc vào thứ tự duyệt.

In [13]:
class SolutionKey:
    def maxDivScore(self, nums: List[int], divisors: List[int]) -> int:
        return max(divisors, key=lambda d: (sum(x % d == 0 for x in nums), -d))

print(SolutionKey().maxDivScore([2, 9, 15, 50], [5, 3, 7, 9]))       # 3
print(SolutionKey().maxDivScore([4, 7, 9, 3, 9], [5, 2, 3]))         # 3
print(SolutionKey().maxDivScore([12], [10, 16]))                     # 10

3
3
10


**Độ phức tạp:** Time `O(n · k)`, Space `O(1)`. Mỗi ước được tính điểm đúng **một** lần (hàm `key` chỉ gọi một lần cho mỗi phần tử).

### Cách 5 — Gom `nums` bằng `Counter`, bỏ ước trùng và ước quá lớn

Hai tối ưu hợp lý khi dữ liệu có nhiều giá trị lặp:

- `cnt = Counter(nums)` cho `{giá trị: số lần}`; điểm của `d` là `sum(c for x, c in cnt.items() if x % d == 0)` — chỉ lặp qua các giá trị **khác nhau**.
- Duyệt `sorted(set(divisors))`: ước trùng chỉ tính một lần; vì tăng dần nên dùng `>` chặt. Ước `d > max(nums)` có điểm `0` nhưng vẫn phải xét vì có thể mọi điểm đều `0` ⇒ đáp án là ước nhỏ nhất.

In [14]:
from collections import Counter

class SolutionCounter:
    def maxDivScore(self, nums: List[int], divisors: List[int]) -> int:
        cnt = Counter(nums)
        best_score, best_d = -1, 0
        for d in sorted(set(divisors)):
            score = sum(c for x, c in cnt.items() if x % d == 0)
            if score > best_score:
                best_score, best_d = score, d
        return best_d

print(SolutionCounter().maxDivScore([2, 9, 15, 50], [5, 3, 7, 9]))       # 3
print(SolutionCounter().maxDivScore([4, 7, 9, 3, 9], [5, 2, 3]))         # 3
print(SolutionCounter().maxDivScore([12], [10, 16]))                     # 10
print(SolutionCounter().maxDivScore([6, 6, 6], [3, 3, 2]))               # 2 (cùng điểm 3; ước nhỏ nhất)

3
3
10
2


**Độ phức tạp:** gọi `u` là số giá trị khác nhau trong `nums`, `w` là số ước khác nhau. Time `O(n + w log w + u · w)`; Space `O(u + w)`. Tệ nhất (mọi số khác nhau) vẫn `O(n · k)`, nhưng nhanh hơn rõ khi có nhiều số trùng.

## 8. TRACE từng bước

Trace Cách 2 trên `nums = [2, 9, 15, 50]`, `divisors = [5, 3, 7, 9]`:

| `d` | các `x` chia hết | `score` | so với kỷ lục `(best_score, best_d)` | kỷ lục sau |
|---|---|---|---|---|
| (khởi tạo) | | | | `(-1, 0)` |
| 5 | `15, 50` | 2 | `2 > -1` ⇒ cập nhật | `(2, 5)` |
| 3 | `9, 15` | 2 | `2 == 2` và `3 < 5` ⇒ cập nhật | `(2, 3)` |
| 7 | (không có) | 0 | `0 < 2` ⇒ giữ | `(2, 3)` |
| 9 | `9` | 1 | `1 < 2` ⇒ giữ | `(2, 3)` |

Kết quả `3`.

In [15]:
def trace_div_score(nums, divisors):
    best_score, best_d = -1, 0
    print("khởi tạo kỷ lục:", (best_score, best_d))
    for d in divisors:
        ok = [x for x in nums if x % d == 0]
        score = len(ok)
        if score > best_score:
            action = f"điểm hơn hẳn ({score} > {best_score}) -> cập nhật"
            best_score, best_d = score, d
        elif score == best_score and d < best_d:
            action = f"điểm bằng ({score}) và {d} < {best_d} -> cập nhật"
            best_score, best_d = score, d
        else:
            action = "giữ nguyên"
        print(f"d={d}: chia hết {ok} -> score={score}; {action}; kỷ lục = {(best_score, best_d)}")
    print("kết quả:", best_d)
    return best_d

trace_div_score([2, 9, 15, 50], [5, 3, 7, 9])

khởi tạo kỷ lục: (-1, 0)
d=5: chia hết [15, 50] -> score=2; điểm hơn hẳn (2 > -1) -> cập nhật; kỷ lục = (2, 5)
d=3: chia hết [9, 15] -> score=2; điểm bằng (2) và 3 < 5 -> cập nhật; kỷ lục = (2, 3)
d=7: chia hết [] -> score=0; giữ nguyên; kỷ lục = (2, 3)
d=9: chia hết [9] -> score=1; giữ nguyên; kỷ lục = (2, 3)
kết quả: 3


3

In [16]:
trace_div_score([12], [10, 16])        # mọi điểm đều 0: ước nhỏ nhất thắng

khởi tạo kỷ lục: (-1, 0)
d=10: chia hết [] -> score=0; điểm hơn hẳn (0 > -1) -> cập nhật; kỷ lục = (0, 10)
d=16: chia hết [] -> score=0; giữ nguyên; kỷ lục = (0, 10)
kết quả: 10


10

## 9. Code nộp LeetCode

Dùng Cách 4: `max` với khóa tuple — ngắn nhất và tie-break rõ ràng. (Có thể thay bằng Cách 2 nếu bạn muốn viết tường minh.)

In [17]:
from typing import List

class Solution:
    def maxDivScore(self, nums: List[int], divisors: List[int]) -> int:
        return max(divisors, key=lambda d: (sum(x % d == 0 for x in nums), -d))

In [18]:
print(Solution().maxDivScore([2, 9, 15, 50], [5, 3, 7, 9]))     # 3
print(Solution().maxDivScore([4, 7, 9, 3, 9], [5, 2, 3]))       # 3
print(Solution().maxDivScore([12], [10, 16]))                   # 10

3
3
10


## 10. Vì sao thuật toán đúng

1. **Điểm tính đúng.** `sum(x % d == 0 for x in nums)` cộng `1` cho mỗi `x` mà `x % d == 0`, tức đúng số chỉ số `j` có `nums[j]` chia hết cho `d`. Vậy `score(d)` khớp định nghĩa của đề.
2. **Khóa tuple.** So sánh tuple là so sánh từ trái sang phải: `(s1, -d1) > (s2, -d2)` khi `s1 > s2`, hoặc `s1 == s2` và `-d1 > -d2` ⟺ `d1 < d2`. Tuple lớn nhất ứng với điểm lớn nhất; trong các ước cùng điểm lớn nhất, tuple lớn nhất ứng với ước **nhỏ nhất** — đúng đề.
3. **Mọi điểm đều `0`.** Khi không ước nào chia hết số nào, mọi tuple có dạng `(0, -d)` ⇒ `max` chọn `-d` lớn nhất tức `d` nhỏ nhất. Đúng tie-break (ví dụ 3).
4. **Ước trùng.** Nếu `divisors` có hai giá trị bằng nhau thì chúng có cùng tuple; `max` trả về một trong hai, và vì chúng bằng nhau nên đáp án (giá trị) như nhau.
5. **Cách 3 và 5 (sắp xếp, `>` chặt).** Duyệt ước tăng dần và chỉ cập nhật khi điểm hơn hẳn: ước đầu tiên đạt max (nhỏ nhất trong các ước đạt max) được giữ lại, các ước sau cùng điểm không thay thế được.

## 11. Phân tích Time / Space Complexity

`n = len(nums)`, `k = len(divisors)`.

| Cách | Time | Space |
|---|---|---|
| 1. Hai vòng lặp | `O(n · k)` | `O(1)` |
| 2. `sum` + kỷ lục | `O(n · k)` | `O(1)` |
| 3. Sắp xếp + `>` chặt | `O(k log k + n · k)` | `O(k)` |
| 4. `max` với khóa tuple | `O(n · k)` | `O(1)` |
| 5. `Counter` + `set` | `O(n + k log k + u · w)` | `O(u + w)` |

Số phép `%` tối đa `1000 × 1000 = 10^6` — chạy trong chưa đến một giây ở Python. Không cần chia nhỏ hơn.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: một phần tử; mọi điểm bằng `0`; hoà nhiều ước; ước `1` (chia hết mọi số); ước lặp; ước lớn hơn mọi số; hoà mà ước nhỏ nhất đứng cuối mảng (kiểm tra tie-break không phụ thuộc thứ tự).

In [19]:
sols = [SolutionNested(), SolutionSum(), SolutionSorted(), SolutionKey(), SolutionCounter(), Solution()]
cases = [
    ("ví dụ 1", [2, 9, 15, 50], [5, 3, 7, 9], 3),
    ("ví dụ 2", [4, 7, 9, 3, 9], [5, 2, 3], 3),
    ("ví dụ 3", [12], [10, 16], 10),
    ("1 phần tử, ước chia hết", [10], [5], 5),
    ("1 phần tử, ước không chia hết", [10], [3], 3),
    ("ước 1 chia hết tất cả", [7, 11, 13], [1, 7, 11], 1),
    ("hoà, ước nhỏ nhất đứng cuối", [6, 12, 18, 5], [6, 3, 2], 2),
    ("hoà, ước nhỏ nhất đứng đầu", [6, 12, 18, 5], [2, 3, 6], 2),
    ("ước lặp", [6, 6, 6], [3, 3, 2], 2),
    ("mọi điểm 0, chọn nhỏ nhất", [1, 1, 1], [9, 5, 7], 5),
    ("ước lớn hơn mọi số", [2, 3, 4], [10**9, 2, 3], 2),
    ("số lớn", [10**9, 10**9 - 1], [10**9, 10**9 - 1, 1], 1),
]
for name, nums, divs, exp in cases:
    for s in sols:
        got = s.maxDivScore(nums[:], divs[:])
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 3
PASS ví dụ 2 -> 3
PASS ví dụ 3 -> 10
PASS 1 phần tử, ước chia hết -> 5
PASS 1 phần tử, ước không chia hết -> 3
PASS ước 1 chia hết tất cả -> 1
PASS hoà, ước nhỏ nhất đứng cuối -> 2
PASS hoà, ước nhỏ nhất đứng đầu -> 2
PASS ước lặp -> 2
PASS mọi điểm 0, chọn nhỏ nhất -> 5
PASS ước lớn hơn mọi số -> 2
PASS số lớn -> 1


Chú ý ca cuối: `10^9` chỉ chia hết `10^9` (điểm 1); `10^9 - 1` chỉ chia hết `10^9 - 1` (điểm 1); `1` chia hết cả hai (điểm 2) ⇒ đáp án `1`.

### 12.2. Phản ví dụ cho lời giải sai thiếu tie-break

Một lời giải sai hay gặp là chỉ cập nhật khi `>` mà **không sắp xếp**. Ta chứng minh ca "hoà, ước nhỏ nhất đứng cuối" bắt được lỗi này.

In [20]:
def sai_khong_tie_break(nums, divisors):
    best_score, best_d = -1, 0
    for d in divisors:                        # không sắp xếp!
        score = sum(x % d == 0 for x in nums)
        if score > best_score:
            best_score, best_d = score, d
    return best_d

print(sai_khong_tie_break([6, 12, 18, 5], [6, 3, 2]), "vs đúng", Solution().maxDivScore([6, 12, 18, 5], [6, 3, 2]))
assert sai_khong_tie_break([6, 12, 18, 5], [6, 3, 2]) == 6       # sai: đáng lẽ là 2
assert sai_khong_tie_break([6, 12, 18, 5], [2, 3, 6]) == 2       # trùng hợp đúng khi đã tăng dần

6 vs đúng 2


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh dữ liệu đúng constraints (`1 <= n, k <= 1000`, giá trị `1..10^9`). Để có nhiều lần chia hết, ta chọn giá trị nhỏ hoặc là bội của một số cho trước. Oracle: tính điểm mọi ước bằng từ điển, lấy `min` các ước đạt max.

In [21]:
import random
random.seed(2644)

def oracle(nums, divisors):
    score = {}
    for d in divisors:
        score[d] = sum(1 for x in nums if x % d == 0)
    top = max(score.values())
    return min(d for d in score if score[d] == top)

for _ in range(500):
    n, k = random.randint(1, 12), random.randint(1, 8)
    hi = random.choice([5, 20, 100, 10**9])
    nums = [random.randint(1, hi) for _ in range(n)]
    divs = [random.randint(1, hi) for _ in range(k)]
    exp = oracle(nums, divs)
    for s in sols:
        got = s.maxDivScore(nums[:], divs[:])
        assert got == exp, (type(s).__name__, nums, divs, got, exp)
print("500 dữ liệu ngẫu nhiên: mọi cách đều khớp oracle")

# kích thước tối đa
nums = [random.randint(1, 1000) for _ in range(1000)]
divs = [random.randint(1, 1000) for _ in range(1000)]
exp = oracle(nums, divs)
assert all(s.maxDivScore(nums[:], divs[:]) == exp for s in sols)
print("1000 x 1000 OK, đáp án:", exp)

500 dữ liệu ngẫu nhiên: mọi cách đều khớp oracle


1000 x 1000 OK, đáp án: 1


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên quy tắc "hoà thì chọn ước nhỏ nhất"

In [22]:
from typing import List

def sai_quen_hoa(nums: List[int], divisors: List[int]) -> int:
    best_score, best_d = -1, 0
    for d in divisors:
        score = sum(x % d == 0 for x in nums)
        if score >= best_score:              # hoà thì lấy ước SAU, không quan tâm nhỏ/lớn
            best_score, best_d = score, d
    return best_d

print("sai :", sai_quen_hoa([2, 9, 15, 50], [5, 3, 7, 9]))      # 3: tình cờ đúng (xem giải thích)
print("đúng:", Solution().maxDivScore([2, 9, 15, 50], [5, 3, 7, 9]))   # 3

sai : 3
đúng: 3


**Vì sao sai:** với `>=`, ước **đứng sau** thay ước đứng trước mỗi khi hoà, nên kết quả phụ thuộc **thứ tự trong `divisors`** chứ không phải độ lớn của ước. Ví dụ 1 có điểm `[2, 2, 0, 1]` cho `[5, 3, 7, 9]`: `3` đứng sau `5` nên **tình cờ** thắng — ca này không bắt được lỗi. Nhưng với `divisors = [3, 5]` thì `5` đứng sau và thắng, sai (đúng phải là `3`). Đề yêu cầu **nhỏ nhất**, nên phải so sánh `d < best_d`, sắp xếp trước, hoặc dùng khóa `-d`.

In [23]:
print("sai :", sai_quen_hoa([2, 9, 15, 50], [3, 5]))            # 5
print("đúng:", Solution().maxDivScore([2, 9, 15, 50], [3, 5]))  # 3

sai : 5
đúng: 3


### Lỗi 2: Đảo vai trò khi dùng `%`

In [24]:
def sai_dao_vai_tro(nums: List[int], divisors: List[int]) -> int:
    return max(divisors, key=lambda d: (sum(d % x == 0 for x in nums), -d))   # d % x thay vì x % d

print("sai :", sai_dao_vai_tro([2, 9, 15, 50], [5, 3, 7, 9]))
print("đúng:", Solution().maxDivScore([2, 9, 15, 50], [5, 3, 7, 9]))
print("sai :", sai_dao_vai_tro([12], [10, 16]), "| đúng:", Solution().maxDivScore([12], [10, 16]))

sai : 9
đúng: 3
sai : 10 | đúng: 10


**Vì sao sai:** `d % x == 0` hỏi "`d` chia hết cho `x`", ngược với đề ("`x` chia hết cho `d`"). Với ví dụ 1, code sai cho kết quả `9` (vì `9 % 9 == 0` là phép duy nhất đúng) thay vì `3`. Với ví dụ 3 (`[12]`, `[10, 16]`) nó trùng đáp án chỉ vì mọi điểm đều `0`. Nhớ: **số bị chia nằm bên trái `%`**.

In [25]:
print("sai :", sai_dao_vai_tro([12], [24, 5]))       # 24 (24 % 12 == 0)
print("đúng:", Solution().maxDivScore([12], [24, 5]))   # 5: không ước nào chia hết 12 -> chọn nhỏ nhất

sai : 24
đúng: 5


### Lỗi 3: Khởi tạo kỷ lục `best_score = 0` làm hỏng trường hợp mọi điểm đều `0`

In [26]:
def sai_khoi_tao(nums: List[int], divisors: List[int]) -> int:
    best_score, best_d = 0, 0              # 0 = "chưa có", nhưng điểm 0 cũng là điểm thật
    for d in sorted(divisors):
        score = sum(x % d == 0 for x in nums)
        if score > best_score:
            best_score, best_d = score, d
    return best_d

print("sai :", sai_khoi_tao([12], [10, 16]))              # 0 -- không phải một ước nào
print("đúng:", Solution().maxDivScore([12], [10, 16]))    # 10

sai : 0
đúng: 10


**Vì sao sai:** khi mọi điểm đều `0`, điều kiện `score > 0` không bao giờ đúng nên `best_d` giữ `0`, mà `0` không nằm trong `divisors`. Khởi tạo `best_score = -1` để ước đầu tiên luôn được nhận, hoặc dùng `max(... key=...)`.

### Lỗi 4: Chọn ước **lớn** nhất khi hoà vì dùng `max` hai lần

In [27]:
def sai_max_hai_lan(nums: List[int], divisors: List[int]) -> int:
    scores = [sum(x % d == 0 for x in nums) for d in divisors]
    top = max(scores)
    return max(d for d, s in zip(divisors, scores) if s == top)    # max thay vì min

print("sai :", sai_max_hai_lan([2, 9, 15, 50], [5, 3, 7, 9]))      # 5
print("đúng:", Solution().maxDivScore([2, 9, 15, 50], [5, 3, 7, 9]))   # 3

sai : 5
đúng: 3


**Vì sao sai:** sau khi lọc các ước đạt điểm cao nhất, đề yêu cầu `min`, không phải `max`. Hai chữ này dễ nhầm khi vừa viết `max(scores)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có hai tiêu chí "lớn nhất theo A, hoà thì nhỏ nhất theo B" ⇒ khóa tuple `(A, -B)` với `max`, hoặc `(-A, B)` với `min`.
- Đếm số phần tử thoả điều kiện ⇒ `sum(điều_kiện for x in ...)`.
- Chia hết ⇒ `x % d == 0` (số bị chia bên trái). Nhớ ví dụ `15 % 5`.
- Khi có thể có "mọi thứ bằng nhau" (mọi điểm `0`), kiểm tra lời giải của bạn trả gì.
- Nhìn ràng buộc: `n, k <= 1000` ⇒ `O(n · k)` là đủ; đừng tìm thuật toán phức tạp hơn.

## 15. Biến thể của bài

1. **Điểm chia hết lớn nhất, trả về cả điểm:** `[ước, điểm]`.
2. **Chọn ước lớn nhất khi hoà:** đổi khóa thành `(điểm, d)`.
3. **Count Number of Pairs With Absolute Difference K / Count Pairs Divisible by K:** thêm yếu tố đếm cặp.
4. **Greatest Common Divisor of Strings / Array** (LeetCode 1979): ước chung lớn nhất của mảng.
5. **Divisible and Non-divisible Sums Difference (LeetCode 2894):** tổng các số chia hết và không chia hết cho `m`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng lặp | Đếm tay, kỷ lục có tie-break | `O(n · k)` | `O(1)` | Người mới |
| 2. `sum` + kỷ lục | Đếm bằng `sum` | `O(n · k)` | `O(1)` | Tường minh, dễ gỡ lỗi |
| 3. Sắp xếp + `>` chặt | Ước nhỏ nhất tới trước | `O(k log k + n · k)` | `O(k)` | Tie-break "miễn phí" |
| 4. `max` + khóa tuple | `(điểm, -ước)` | `O(n · k)` | `O(1)` | **Nên nộp**: ngắn nhất |
| 5. `Counter` + `set` | Gom giá trị trùng | `O(u · w)` | `O(u + w)` | Dữ liệu có nhiều giá trị lặp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `15 % 5`, `9 % 2 == 0`, `sum([True, False, True])`, `max([3, 5, 7], key=lambda d: -d)`.

**Bài 2 (tính tay).** Với `nums = [6, 12, 18, 5]`, `divisors = [2, 3, 6]`, tính điểm từng ước và đáp án của `maxDivScore`.

**Bài 3 (điền TODO).** Hoàn thành `div_score_todo(nums, d)` trả điểm chia hết của `d`.

In [28]:
def div_score_todo(nums, d):
    score = 0
    # TODO 1: duyệt từng x trong nums
    # TODO 2: nếu x chia hết cho d (x % d == 0) thì tăng score
    # TODO 3: return score
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `max_div_score_key(nums, divisors)` chỉ bằng một lệnh `return max(..., key=...)` mà không nhìn Cách 4. Kiểm tra với ví dụ 1 và 3.

**Bài 5 (biến thể / mini challenge).** Viết `all_best_divisors(nums, divisors)` trả về **danh sách các ước (không trùng, tăng dần)** đạt điểm cao nhất. Mini-challenge thêm: `max_div_score_largest(nums, divisors)` — khi hoà chọn ước **lớn nhất**.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `15 % 5 = 0`; `9 % 2 = 1` nên `False`; `True` là `1`; `key=lambda d: -d` làm số nhỏ nhất thành "lớn nhất". Bài 2: `2` chia hết `6, 12, 18`; `3` chia hết `6, 12, 18`; `6` chia hết `6, 12, 18`; `5` không chia hết bởi cả ba.
- **Mức 2 (vừa):** Bài 3: `for x in nums: if x % d == 0: score += 1`. Bài 4: khóa `lambda d: (sum(x % d == 0 for x in nums), -d)`.
- **Mức 3 (sâu):** Bài 5: tính từ điển `{d: điểm}` cho `set(divisors)`, lấy `top = max(...)`, rồi `sorted(d for d, s in scores.items() if s == top)`. Với "lớn nhất khi hoà", đổi khóa thành `(điểm, d)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert 15 % 5 == 0
assert (9 % 2 == 0) is False
assert sum([True, False, True]) == 2
assert max([3, 5, 7], key=lambda d: -d) == 3

# Bài 2
nums, divs = [6, 12, 18, 5], [2, 3, 6]
assert {d: sum(x % d == 0 for x in nums) for d in divs} == {2: 3, 3: 3, 6: 3}
assert Solution().maxDivScore(nums, divs) == 2

# Bài 3
def div_score_ans(nums, d):
    score = 0
    for x in nums:
        if x % d == 0:
            score += 1
    return score

assert div_score_ans([2, 9, 15, 50], 5) == 2
assert div_score_ans([2, 9, 15, 50], 7) == 0

# Bài 4
def max_div_score_key(nums, divisors):
    return max(divisors, key=lambda d: (sum(x % d == 0 for x in nums), -d))

assert max_div_score_key([2, 9, 15, 50], [5, 3, 7, 9]) == 3
assert max_div_score_key([12], [10, 16]) == 10

# Bài 5
def all_best_divisors(nums, divisors):
    scores = {d: sum(x % d == 0 for x in nums) for d in set(divisors)}
    top = max(scores.values())
    return sorted(d for d, s in scores.items() if s == top)

assert all_best_divisors([2, 9, 15, 50], [5, 3, 7, 9]) == [3, 5]
assert all_best_divisors([12], [10, 16, 10]) == [10, 16]
assert all_best_divisors([4, 7, 9, 3, 9], [5, 2, 3]) == [3]

def max_div_score_largest(nums, divisors):
    return max(divisors, key=lambda d: (sum(x % d == 0 for x in nums), d))

assert max_div_score_largest([2, 9, 15, 50], [5, 3, 7, 9]) == 5
assert max_div_score_largest([12], [10, 16]) == 16
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (khóa tuple: điểm lớn, ước nhỏ)

```python
from typing import List

class Solution:
    def maxDivScore(self, nums: List[int], divisors: List[int]) -> int:
        return max(divisors, key=lambda d: (sum(x % d == 0 for x in nums), -d))
```

**Checklist**

- [ ] Điểm của `d` đếm các `x` với `x % d == 0` (số bị chia ở bên **trái**)?
- [ ] Hoà thì chọn ước **nhỏ nhất** (khóa `-d`, hoặc so `d < best_d`, hoặc sắp xếp trước)?
- [ ] Mọi điểm đều `0` vẫn trả ước nhỏ nhất của `divisors` (không trả `0`)?
- [ ] Ước lặp, ước `1`, ước lớn hơn mọi số đã được thử?
- [ ] Kích thước `1000 × 1000` chạy nhanh (`O(n · k)`)?
- [ ] Signature đúng `maxDivScore(self, nums: List[int], divisors: List[int]) -> int` và có `from typing import List`?